# 🚬 Train an Accurate Cigarette Detector (Google Colab / Kaggle, Free GPU)

Open this notebook in **Google Colab**, set **Runtime → Change runtime type → GPU (T4)**, and **Run all**.
In ~1 hour you will download a precision-tuned `best.pt` model that detects cigarettes and **NOT** fingers/pens.

### Why current models call small objects "cigarettes":
They were trained *only* on cigarette photos, so they learned `thin elongated blob = cigarette`.

### Fixes baked into this notebook:
1. **Hard Negatives** — Images of fingers/pens/hands with NO cigarette labels. YOLO treats label-less images as background, learning to stay quiet on non-cigarette objects.
2. **High Resolution** (`imgsz: 1280`) — Preserves fine details for tiny cigarette objects.
3. **Confidence Sweep** — Finds and recommends the exact threshold to set in `config.yaml`.

In [ ]:
# 1) Check GPU availability (Tesla T4 / similar)
!nvidia-smi

# 2) Install required packages
!pip -q install ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3) Download a Labelled Cigarette Dataset

Open one of these Roboflow Universe datasets, click **Download Dataset → YOLOv11 → Show Download Code**, and paste your snippet in the cell below:
- [Smoker YOLO (~4100 imgs)](https://universe.roboflow.com/cigaretteple-7m0hn/smoker-yolo)
- [Cigarette (~4900 imgs)](https://universe.roboflow.com/cigarette-c6554/cigarette-ghnlk)
- [Cigarette Detection](https://universe.roboflow.com/cigarettedetection-bxzzm/cigarette-detection-qgqhn)

In [ ]:
# 3) Download dataset — REPLACE this block with your Roboflow snippet
from roboflow import Roboflow
rf = Roboflow(api_key="PASTE_YOUR_FREE_API_KEY")
project = rf.workspace("cigaretteple-7m0hn").project("smoker-yolo")
dataset = project.version(1).download("yolov11")   # Use version number shown on Roboflow

DATA_YAML = dataset.location + "/data.yaml"
print("data.yaml:", DATA_YAML)
!cat "$DATA_YAML"

## 4) Add HARD NEGATIVES (Stop Finger / Pen False Positives)

Upload 100–300 photos of objects wrongly flagged (**fingers, pens, straws, pointing hands**) with **NO** cigarettes in them.
In Colab: click the folder icon on the left panel, create a folder `negatives`, and upload your images there.
This cell copies them into the dataset with empty `.txt` labels (= background images).

In [ ]:
# 4) Import hard negatives with EMPTY labels
import os, glob, shutil, yaml
NEG_DIR = "negatives"   # Folder filled with finger/pen/straw photos

with open(DATA_YAML) as f:
    dcfg = yaml.safe_load(f)
root = dcfg.get("path", os.path.dirname(DATA_YAML))
train_img = os.path.join(root, dcfg.get("train", "train/images"))
train_lbl = train_img.replace("images", "labels")
os.makedirs(train_lbl, exist_ok=True)

added = 0
if os.path.isdir(NEG_DIR):
    for p in glob.glob(NEG_DIR + "/*"):
        if p.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp")):
            b = os.path.basename(p)
            stem = os.path.splitext(b)[0]
            shutil.copy(p, os.path.join(train_img, "neg_" + b))
            open(os.path.join(train_lbl, "neg_" + stem + ".txt"), "w").close()  # Empty = background
            added += 1
print(f"Added {added} hard-negative images. (0 = skipped — add images to eliminate false positives.)")

## 5) Precision-Focused Training

High resolution (`imgsz: 1280`), Cosine LR, Mosaic disabled for the final 15 epochs, and raised classification loss (`cls: 1.0`) to strictly enforce class verification.

In [ ]:
# 5) Train
from ultralytics import YOLO
model = YOLO("yolo11m.pt")   # Use yolo11s.pt if out of memory
model.train(
    data=DATA_YAML,
    epochs=150,
    imgsz=1280,          # Drop to 960 if OOM
    batch=8,             # Drop to 4 if OOM
    patience=40,
    cos_lr=True, lr0=0.01, warmup_epochs=3.0,
    box=7.5, cls=1.0, dfl=1.5,             # cls raised -> stricter "is this really a cigarette?"
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.5, fliplr=0.5,
    mosaic=1.0, close_mosaic=15, mixup=0.05, copy_paste=0.05,
    plots=True, name="cig_precision",
)

## 6) Validate & Confidence Threshold Sweep

In [ ]:
# 6) Validate + find the confidence threshold that keeps precision high
m = model.val(data=DATA_YAML, imgsz=1280)
print(f"mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}")

print("\nconf  precision  recall")
best = None
for conf in (0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60):
    mv = model.val(data=DATA_YAML, imgsz=1280, conf=conf, verbose=False)
    p, r = float(mv.box.mp), float(mv.box.mr)
    print(f"{conf:.2f}   {p:0.3f}      {r:0.3f}")
    key = (p >= 0.90, r if p >= 0.90 else 0.0)
    if best is None or key > best[0]: best = (key, conf, p, r)
if best:
    print(f"\n>>> Set detection.cig_conf = {best[1]:.2f} in config.yaml (precision {best[2]:.2f}, recall {best[3]:.2f})")

## 7) Download Trained `best.pt` Model

In [ ]:
# 7) Download your trained model
import shutil
shutil.copy("runs/detect/cig_precision/weights/best.pt", "best.pt")
try:
    from google.colab import files
    files.download("best.pt")   # Saves best.pt to your local machine
except Exception:
    print("Not on Colab — grab best.pt from runs/detect/cig_precision/weights/best.pt")